# Model properties vs performance — correlation analysis

This notebook answers: **which measurable model traits move together with load time, FPS, and memory?**

It is a focused correlation study (not a full predictive modeling pipeline). Use it to prioritize which DensityStats / geometry fields are worth deeper investigation before changing rendering or loading code.

## Inputs

| Source | Files |
|--------|--------|
| **Demo (default)** | `model-correlation-sample-data/models_merged.csv` |
| **Split demo** | `model_properties.csv` + `model_performance.csv` (same folder) |
| **Custom** | Merge benchmark CSV with DensityStats table on `model_name` or `urn` |

Demo CSVs live under `model-correlation-sample-data/` (paths are relative to the **Science** folder).

## Method

- **Spearman rank correlation (ρ)** — robust when counts are heavy-tailed and monotonic but non-linear.
- **Heatmap** — properties (rows) vs outcomes (columns).
- **Top pairs** — strongest |ρ| per outcome with scatter plots.

**Caution:** correlation is not causation; keep app version and device fixed when comparing builds.

## Configuration

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from model_correlation_utils import (
    DEFAULT_OUTCOME_COLUMNS,
    DEFAULT_PROPERTY_COLUMNS,
    columns_present,
    spearman_correlation_table,
    spearman_matrix,
    spearman_pair,
    top_correlations_for_outcome,
)

SCIENCE_ROOT = Path.cwd().resolve()
DEMO_DIR = SCIENCE_ROOT / "model-correlation-sample-data"

# "merged" | "split" | "custom"
DATA_SOURCE = "merged"
CUSTOM_MERGED_CSV: Path | None = None  # e.g. SCIENCE_ROOT / "model-correlation-sample-data" / "my_merged.csv"

PRIMARY_OUTCOME = "init_time_ms"  # also: fps, stage_duration_sec, peak_memory_kb
MIN_SAMPLES = 5
SHOW_PLOTS = True

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (9, 5)

## 1. Load data

In [ ]:
def load_analysis_frame() -> pd.DataFrame:
    if DATA_SOURCE == "split":
        props = pd.read_csv(DEMO_DIR / "model_properties.csv")
        perf = pd.read_csv(DEMO_DIR / "model_performance.csv")
        return props.merge(perf, on=["model_name", "urn"], how="inner")
    if DATA_SOURCE == "merged":
        path = DEMO_DIR / "models_merged.csv"
    elif DATA_SOURCE == "custom":
        if CUSTOM_MERGED_CSV is None or not CUSTOM_MERGED_CSV.is_file():
            raise FileNotFoundError("Set CUSTOM_MERGED_CSV to a merged property+performance table.")
        path = CUSTOM_MERGED_CSV
    else:
        raise ValueError(f"Unknown DATA_SOURCE: {DATA_SOURCE}")
    if not path.is_file():
        raise FileNotFoundError(
            f"Missing {path}. Expected demo files under {DEMO_DIR} (run notebook with cwd = Science)."
        )
    return pd.read_csv(path)


df = load_analysis_frame()
print(f"Models: {len(df)}, columns: {len(df.columns)}")
df.head()

In [ ]:
property_cols = columns_present(df, DEFAULT_PROPERTY_COLUMNS)
outcome_cols = columns_present(df, DEFAULT_OUTCOME_COLUMNS)

if PRIMARY_OUTCOME not in outcome_cols:
    raise ValueError(f"PRIMARY_OUTCOME {PRIMARY_OUTCOME!r} not in data. Found: {outcome_cols}")

analysis = df.dropna(subset=[PRIMARY_OUTCOME]).copy()
print("Properties:", property_cols)
print("Outcomes:", outcome_cols)
analysis[["model_name", PRIMARY_OUTCOME] + property_cols[:3]].describe()

## 2. Spearman correlations (all property × outcome pairs)

In [ ]:
corr_table = spearman_correlation_table(
    analysis,
    property_cols,
    outcome_cols,
    min_samples=MIN_SAMPLES,
)

top_for_primary = top_correlations_for_outcome(corr_table, PRIMARY_OUTCOME)
top_for_primary

In [ ]:
if SHOW_PLOTS and not corr_table.empty:
    heatmap_cols = [c for c in property_cols if analysis[c].notna().sum() >= MIN_SAMPLES]
    if heatmap_cols and outcome_cols:
        rho = spearman_matrix(analysis, heatmap_cols, outcome_cols)
        plt.figure(figsize=(9, max(4, 0.4 * len(heatmap_cols))))
        sns.heatmap(
            rho,
            annot=True,
            fmt=".2f",
            cmap="coolwarm",
            center=0,
            vmin=-1,
            vmax=1,
        )
        plt.title("Spearman ρ: model properties vs performance outcomes")
        plt.tight_layout()
        plt.show()

## 3. Strongest drivers for the primary outcome

Bar chart of |ρ| for `PRIMARY_OUTCOME`, then scatter plots for the top properties.

In [ ]:
if SHOW_PLOTS and not top_for_primary.empty:
    plot_df = top_for_primary.sort_values("abs_rho", ascending=True)
    plt.figure(figsize=(8, 4))
    colors = plot_df["spearman_rho"].apply(lambda v: "#dc2626" if v > 0 else "#2563eb")
    plt.barh(plot_df["property"], plot_df["abs_rho"], color=colors)
    plt.xlabel("|Spearman ρ|")
    plt.title(f"Strongest property associations with {PRIMARY_OUTCOME}")
    plt.tight_layout()
    plt.show()

In [ ]:
top_props = top_for_primary["property"].head(4).tolist() if not top_for_primary.empty else property_cols[:4]

if SHOW_PLOTS and top_props:
    fig, axes = plt.subplots(2, 2, figsize=(10, 8))
    for ax, prop in zip(axes.flatten(), top_props, strict=False):
        pair = analysis[[prop, PRIMARY_OUTCOME]].dropna()
        sns.regplot(
            data=pair,
            x=prop,
            y=PRIMARY_OUTCOME,
            scatter_kws={"alpha": 0.75, "s": 35},
            line_kws={"color": "#0f766e"},
            ax=ax,
        )
        rho, _, _ = spearman_pair(pair[prop], pair[PRIMARY_OUTCOME])
        ax.set_title(f"{PRIMARY_OUTCOME} vs {prop} (ρ={rho:.2f})")
    plt.tight_layout()
    plt.show()

## 4. Export correlation table

Save results next to the input data for sharing or Power BI.

In [ ]:
WRITE_CSV = True

if WRITE_CSV and not corr_table.empty:
    out_path = DEMO_DIR / "correlation_results.csv"
    corr_table.to_csv(out_path, index=False)
    print(f"Wrote {out_path}")

## 5. Using your own data

1. Collect DensityStats (or equivalent geometry scalars) and benchmark metrics for each model.
2. Build a table with one row per model: property scalars + performance columns, joined on `model_name` or `urn`.
3. Set `DATA_SOURCE = "custom"` and `CUSTOM_MERGED_CSV` to that file.

For a full EDA + Ridge / random forest workflow, see `model-performance-analysis/notebooks/model_properties_vs_performance.ipynb`.